In [1]:
import pandas as pd

# Load the Zillow dataset
zhvi_data = pd.read_csv("../data/raw/zillow/County_zhvi_uc_sfrcondo_tier_0.33_0.67_sm_sa_month.csv")

# Load the FEMA NRI dataset
nri_data = pd.read_csv("../data/raw/fema_nri/NRI_Table_Counties.csv")

# Display the first few rows to inspect the data
print("Zillow Data - First Rows:")
print(zhvi_data.head())

print("FEMA NRI Data - First Rows:")
print(nri_data.head())

Zillow Data - First Rows:
   RegionID  SizeRank          RegionName RegionType StateName State  \
0      3101         0  Los Angeles County     county        CA    CA   
1       139         1         Cook County     county        IL    IL   
2      1090         2       Harris County     county        TX    TX   
3      2402         3     Maricopa County     county        AZ    AZ   
4      2841         4    San Diego County     county        CA    CA   

                                  Metro  StateCodeFIPS  MunicipalCodeFIPS  \
0    Los Angeles-Long Beach-Anaheim, CA              6                 37   
1    Chicago-Naperville-Elgin, IL-IN-WI             17                 31   
2  Houston-The Woodlands-Sugar Land, TX             48                201   
3             Phoenix-Mesa-Chandler, AZ              4                 13   
4    San Diego-Chula Vista-Carlsbad, CA              6                 73   

      2000-01-31  ...     2025-04-30     2025-05-31     2025-06-30  \
0  20737

In [2]:
# Check for missing values in both datasets
print(zhvi_data.isnull().sum())
print(nri_data.isnull().sum())

# Inspect data types
print(zhvi_data.info())
print(nri_data.info())

RegionID      0
SizeRank      0
RegionName    0
RegionType    0
StateName     0
             ..
2025-09-30    1
2025-10-31    0
2025-11-30    0
2025-12-31    0
2026-01-31    0
Length: 322, dtype: int64
OID_               0
NRI_ID             0
STATE              0
STATEABBRV         0
STATEFIPS          0
                  ..
WNTW_ALR_NPCTL     2
WNTW_RISKV        90
WNTW_RISKS        90
WNTW_RISKR         0
NRI_VER            0
Length: 465, dtype: int64
<class 'pandas.DataFrame'>
RangeIndex: 3073 entries, 0 to 3072
Columns: 322 entries, RegionID to 2026-01-31
dtypes: float64(313), int64(4), str(5)
memory usage: 7.5 MB
None
<class 'pandas.DataFrame'>
RangeIndex: 3232 entries, 0 to 3231
Columns: 465 entries, OID_ to NRI_VER
dtypes: float64(396), int64(5), str(64)
memory usage: 11.5 MB
None


In [8]:
# List columns to identify relevant ones (e.g., RegionID, StateCodeFIPS, and monthly housing price columns)
print("Zillow Columns:")
print(zhvi_data.columns[:20])  # Show the first 20 columns to inspect

# Display the first 5 rows of the first 20 columns
print("Zillow Data (First 5 Rows of First 20 Columns):")
print(zhvi_data.iloc[:5, :20])  # First 5 rows, first 20 columns

Zillow Columns:
Index(['RegionID', 'SizeRank', 'RegionName', 'RegionType', 'StateName',
       'State', 'Metro', 'StateCodeFIPS', 'MunicipalCodeFIPS', '2000-01-31',
       '2000-02-29', '2000-03-31', '2000-04-30', '2000-05-31', '2000-06-30',
       '2000-07-31', '2000-08-31', '2000-09-30', '2000-10-31', '2000-11-30'],
      dtype='str')
Zillow Data (First 5 Rows of First 20 Columns):
   RegionID  SizeRank          RegionName RegionType StateName State  \
0      3101         0  Los Angeles County     county        CA    CA   
1       139         1         Cook County     county        IL    IL   
2      1090         2       Harris County     county        TX    TX   
3      2402         3     Maricopa County     county        AZ    AZ   
4      2841         4    San Diego County     county        CA    CA   

                                  Metro  StateCodeFIPS  MunicipalCodeFIPS  \
0    Los Angeles-Long Beach-Anaheim, CA              6                 37   
1    Chicago-Naperville-El

In [11]:
# Select only the relevant columns
required_columns = ['RegionID', 'RegionName', 'State', 'StateCodeFIPS'] + [col for col in zhvi_data.columns if '2010' <= col[:4] <= '2025']

# Extract relevant data
zhvi_filtered = zhvi_data[required_columns]

# Display the first 5 rows
print(zhvi_filtered.head())

# Save the filtered Zillow data in the 'interim' folder
zhvi_filtered.to_csv('../data/interim/filtered_zhvi_data.csv', index=False)

   RegionID          RegionName State  StateCodeFIPS     2010-01-31  \
0      3101  Los Angeles County    CA              6  375563.833049   
1       139         Cook County    IL             17  197131.708340   
2      1090       Harris County    TX             48  133925.921606   
3      2402     Maricopa County    AZ              4  165359.081575   
4      2841    San Diego County    CA              6  354111.962698   

      2010-02-28     2010-03-31     2010-04-30     2010-05-31     2010-06-30  \
0  375860.505000  376455.199604  378149.178139  381377.663787  381365.872620   
1  196891.053975  195303.317792  195673.201573  195028.449389  195688.652879   
2  133842.258867  133936.727151  134336.507037  134688.043247  134626.154716   
3  164540.503081  163252.224115  162681.446579  162069.921777  161502.520401   
4  356122.259857  358097.280099  361285.699670  362602.954785  362788.672359   

   ...     2025-03-31     2025-04-30     2025-05-31     2025-06-30  \
0  ...  872914.796626 

In [14]:
# Load the filtered Zillow dataset
filtered_zhvi_data = pd.read_csv("../data/interim/filtered_zhvi_data.csv")

# Melt the Zillow dataset from wide format to long format
zhvi_long = filtered_zhvi_data.melt(id_vars=["RegionID", "RegionName", "State", "StateCodeFIPS"], 
                                    var_name="Date", 
                                    value_name="HousingPrice")

# Convert 'Date' column to datetime type
zhvi_long['Date'] = pd.to_datetime(zhvi_long['Date'], format='%Y-%m-%d')

# Inspect the reshaped data
print(zhvi_long.head())

zhvi_long.to_csv('../data/interim/reshaped_zhvi_data.csv', index=False)

   RegionID          RegionName State  StateCodeFIPS       Date   HousingPrice
0      3101  Los Angeles County    CA              6 2010-01-31  375563.833049
1       139         Cook County    IL             17 2010-01-31  197131.708340
2      1090       Harris County    TX             48 2010-01-31  133925.921606
3      2402     Maricopa County    AZ              4 2010-01-31  165359.081575
4      2841    San Diego County    CA              6 2010-01-31  354111.962698


In [19]:
# Load the reshaped Zillow dataset
reshaped_zhvi_data = pd.read_csv("../data/interim/reshaped_zhvi_data.csv")

# Filter the data for Florida (StateCodeFIPS for Florida is 12)
florida_zhvi_data = reshaped_zhvi_data[reshaped_zhvi_data['StateCodeFIPS'] == 12]

# Save the filtered data to a new CSV file
florida_zhvi_data.to_csv('../data/interim/florida_zhvi_data.csv', index=False)

# Display the first few rows to verify the filtering
print(florida_zhvi_data.head())

    RegionID           RegionName State  StateCodeFIPS        Date  \
7       2964    Miami-Dade County    FL             12  2010-01-31   
17      1561       Broward County    FL             12  2010-01-31   
25      2993    Palm Beach County    FL             12  2010-01-31   
27      3165  Hillsborough County    FL             12  2010-01-31   
28      1287        Orange County    FL             12  2010-01-31   

     HousingPrice  
7   174162.681875  
17  146697.046873  
25  176963.884806  
27  142764.252688  
28  151282.109366  


In [20]:
# Check for missing values in Zillow data
florida_zhvi_data = pd.read_csv("../data/interim/florida_zhvi_data.csv")
missing_values_zhvi = florida_zhvi_data.isnull().sum()
print("Missing values in Zillow dataset:\n", missing_values_zhvi)

# Check for duplicates in Zillow data
duplicates_zhvi = florida_zhvi_data.duplicated().sum()
print(f"Number of duplicate rows in Zillow dataset: {duplicates_zhvi}")

Missing values in Zillow dataset:
 RegionID          0
RegionName        0
State             0
StateCodeFIPS     0
Date              0
HousingPrice     98
dtype: int64
Number of duplicate rows in Zillow dataset: 0


In [21]:
# Analyze missingness pattern in Florida Zillow data

# 1. Total missing values
print("Missing values by column:")
print(florida_zhvi_data.isnull().sum())

# 2. Percentage missing
missing_pct = florida_zhvi_data.isnull().mean() * 100
print("\nMissing percentage by column:")
print(missing_pct)

# 3. Rows where HousingPrice is missing
missing_housing = florida_zhvi_data[florida_zhvi_data["HousingPrice"].isnull()]

print("\nNumber of missing HousingPrice rows:", len(missing_housing))
print("\nSample rows with missing HousingPrice:")
print(missing_housing.head())

# 4. Missing values by county
missing_by_county = (
    missing_housing.groupby("RegionName")
    .size()
    .sort_values(ascending=False)
)

print("\nMissing HousingPrice values by county:")
print(missing_by_county)

# 5. Missing values by date
missing_by_date = (
    missing_housing.groupby("Date")
    .size()
    .sort_values(ascending=False)
)

print("\nMissing HousingPrice values by date:")
print(missing_by_date)

Missing values by column:
RegionID          0
RegionName        0
State             0
StateCodeFIPS     0
Date              0
HousingPrice     98
dtype: int64

Missing percentage by column:
RegionID         0.000000
RegionName       0.000000
State            0.000000
StateCodeFIPS    0.000000
Date             0.000000
HousingPrice     0.761816
dtype: float64

Number of missing HousingPrice rows: 98

Sample rows with missing HousingPrice:
     RegionID         RegionName State  StateCodeFIPS        Date  \
37       1214      Monroe County    FL             12  2010-01-31   
52       3050  Washington County    FL             12  2010-01-31   
66       1858     Liberty County    FL             12  2010-01-31   
104      1214      Monroe County    FL             12  2010-02-28   
119      3050  Washington County    FL             12  2010-02-28   

     HousingPrice  
37            NaN  
52            NaN  
66            NaN  
104           NaN  
119           NaN  

Missing HousingPrice v

In [22]:
monroe = florida_zhvi_data[florida_zhvi_data["RegionName"] == "Monroe County"]
print(monroe.head(20))

      RegionID     RegionName State  StateCodeFIPS        Date  HousingPrice
37        1214  Monroe County    FL             12  2010-01-31           NaN
104       1214  Monroe County    FL             12  2010-02-28           NaN
171       1214  Monroe County    FL             12  2010-03-31           NaN
238       1214  Monroe County    FL             12  2010-04-30           NaN
305       1214  Monroe County    FL             12  2010-05-31           NaN
372       1214  Monroe County    FL             12  2010-06-30           NaN
439       1214  Monroe County    FL             12  2010-07-31           NaN
506       1214  Monroe County    FL             12  2010-08-31           NaN
573       1214  Monroe County    FL             12  2010-09-30           NaN
640       1214  Monroe County    FL             12  2010-10-31           NaN
707       1214  Monroe County    FL             12  2010-11-30           NaN
774       1214  Monroe County    FL             12  2010-12-31           NaN

In [23]:
# Find first available value for Monroe County
monroe = florida_zhvi_data[florida_zhvi_data["RegionName"] == "Monroe County"]

print(monroe[monroe["HousingPrice"].notnull()].head())

      RegionID     RegionName State  StateCodeFIPS        Date   HousingPrice
4928      1214  Monroe County    FL             12  2016-02-29  491962.188031
4995      1214  Monroe County    FL             12  2016-03-31  495584.679629
5062      1214  Monroe County    FL             12  2016-04-30  499254.820330
5129      1214  Monroe County    FL             12  2016-05-31  506175.175922
5196      1214  Monroe County    FL             12  2016-06-30  511598.241002


In [24]:
# Total missing values
total_missing = florida_zhvi_data["HousingPrice"].isnull().sum()

# Missing values by county
missing_by_county = (
    florida_zhvi_data[florida_zhvi_data["HousingPrice"].isnull()]
    .groupby("RegionName")
    .size()
    .sort_values(ascending=False)
)

# Convert to DataFrame for better readability
missing_df = missing_by_county.reset_index()
missing_df.columns = ["County", "MissingCount"]

# Calculate percentage contribution
missing_df["Percentage"] = (missing_df["MissingCount"] / total_missing) * 100

# Display results
print("Total missing values:", total_missing)
print("\nMissing values breakdown by county:")
print(missing_df)

Total missing values: 98

Missing values breakdown by county:
              County  MissingCount  Percentage
0      Monroe County            73   74.489796
1     Liberty County            12   12.244898
2  Washington County            12   12.244898
3       Dixie County             1    1.020408


In [25]:
# Inspect Liberty County
liberty = florida_zhvi_data[florida_zhvi_data["RegionName"] == "Liberty County"]

print("Liberty County (first 20 rows):")
print(liberty.head(20))

print("\nLiberty County missing rows:")
print(liberty[liberty["HousingPrice"].isnull()])


# Inspect Washington County
washington = florida_zhvi_data[florida_zhvi_data["RegionName"] == "Washington County"]

print("\nWashington County (first 20 rows):")
print(washington.head(20))

print("\nWashington County missing rows:")
print(washington[washington["HousingPrice"].isnull()])

Liberty County (first 20 rows):
      RegionID      RegionName State  StateCodeFIPS        Date  HousingPrice
66        1858  Liberty County    FL             12  2010-01-31           NaN
133       1858  Liberty County    FL             12  2010-02-28           NaN
200       1858  Liberty County    FL             12  2010-03-31           NaN
267       1858  Liberty County    FL             12  2010-04-30           NaN
334       1858  Liberty County    FL             12  2010-05-31           NaN
401       1858  Liberty County    FL             12  2010-06-30           NaN
468       1858  Liberty County    FL             12  2010-07-31           NaN
535       1858  Liberty County    FL             12  2010-08-31           NaN
602       1858  Liberty County    FL             12  2010-09-30           NaN
669       1858  Liberty County    FL             12  2010-10-31           NaN
736       1858  Liberty County    FL             12  2010-11-30           NaN
803       1858  Liberty County  

In [28]:
# Reload original Florida Zillow data
florida_zhvi_data = pd.read_csv("../data/interim/florida_zhvi_data.csv")

# Convert Date to datetime
florida_zhvi_data["Date"] = pd.to_datetime(florida_zhvi_data["Date"])

# Sort data
florida_zhvi_data = florida_zhvi_data.sort_values(
    by=["RegionName", "Date"]
).reset_index(drop=True)

# Keep rows from the first valid HousingPrice onward for each county
florida_zhvi_data["valid_started"] = (
    florida_zhvi_data
    .groupby("RegionName")["HousingPrice"]
    .transform(lambda x: x.notna().cummax())
)

trimmed_data = florida_zhvi_data[florida_zhvi_data["valid_started"]].copy()

# Drop helper column
trimmed_data = trimmed_data.drop(columns=["valid_started"])

# Check remaining missing rows
remaining_missing = trimmed_data[trimmed_data["HousingPrice"].isnull()]
print("Remaining missing rows:")
print(remaining_missing)

# Interpolate final isolated missing value within county
trimmed_data["HousingPrice"] = (
    trimmed_data
    .groupby("RegionName")["HousingPrice"]
    .transform(lambda x: x.interpolate(method="linear"))
)

# Re-sort final data
trimmed_data = trimmed_data.sort_values(
    by=["Date", "RegionName"]
).reset_index(drop=True)

# Final checks
print("Final shape:", trimmed_data.shape)

print("\nMissing values after final cleaning:")
print(trimmed_data.isnull().sum())

print("\nColumns:")
print(trimmed_data.columns)

# Save cleaned data
trimmed_data.to_csv("../data/interim/cleaned_florida_zhvi_data.csv", index=False)

Remaining missing rows:
      RegionID    RegionName State  StateCodeFIPS       Date  HousingPrice
2527       480  Dixie County    FL             12 2012-08-31           NaN
Final shape: (12767, 6)

Missing values after final cleaning:
RegionID         0
RegionName       0
State            0
StateCodeFIPS    0
Date             0
HousingPrice     0
dtype: int64

Columns:
Index(['RegionID', 'RegionName', 'State', 'StateCodeFIPS', 'Date',
       'HousingPrice'],
      dtype='str')


In [29]:
# =========================
# Final Data Quality Checks
# =========================

# 1. Check for duplicate rows
duplicates = trimmed_data.duplicated().sum()
print(f"Number of duplicate rows: {duplicates}")

# 2. Check for invalid housing prices (<= 0)
invalid_prices = trimmed_data[trimmed_data["HousingPrice"] <= 0]
print(f"\nNumber of invalid (<=0) housing prices: {len(invalid_prices)}")

# 3. Summary statistics (to inspect outliers)
print("\nHousing Price Summary:")
print(trimmed_data["HousingPrice"].describe())

# 4. Check time continuity within each county
date_diffs = trimmed_data.groupby("RegionName")["Date"].diff()
print("\nDate difference distribution (should mostly be ~30/31 days):")
print(date_diffs.value_counts().head())

# 5. Check number of unique counties
num_counties = trimmed_data["RegionName"].nunique()
print(f"\nNumber of unique counties: {num_counties}")

# 6. Final sorting check
trimmed_data = trimmed_data.sort_values(
    by=["RegionName", "Date"]
).reset_index(drop=True)

print("\nFinal dataset shape:", trimmed_data.shape)

# 7. Final column check
print("\nColumns:")
print(trimmed_data.columns)

Number of duplicate rows: 0

Number of invalid (<=0) housing prices: 0

Housing Price Summary:
count    1.276700e+04
mean     2.108995e+05
std      1.173361e+05
min      6.831907e+04
25%      1.265801e+05
50%      1.819608e+05
75%      2.639765e+05
max      1.001663e+06
Name: HousingPrice, dtype: float64

Date difference distribution (should mostly be ~30/31 days):
Date
31 days    7381
30 days    4256
28 days     797
29 days     266
Name: count, dtype: int64

Number of unique counties: 67

Final dataset shape: (12767, 6)

Columns:
Index(['RegionID', 'RegionName', 'State', 'StateCodeFIPS', 'Date',
       'HousingPrice'],
      dtype='str')
